## Run test: Qwen/Qwen3-Reranker-4B

Model page: https://huggingface.co/Qwen/Qwen3-Reranker-4B

Task: `text-ranking`  ·  Library: `transformers`  ·  Source: official-template

> Reproduced from HuggingFace's official auto-generated colab/transformers snippet (identical code). Local inference on 8×W7900 with `device_map="auto"` added.

In [ ]:
%pip install -U transformers accelerate

In [ ]:
# README-aligned reranker inference smoke test.
# HF generated Colab currently loads AutoModelForMultimodalLM only; the model
# card recommends CausalLM scoring for direct Transformers reranking.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/disk/ssd1/zihaomu_amd/models/Qwen__Qwen3-Reranker-4B"
tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_path).cuda().eval()

def format_instruction(instruction, query, doc):
    if instruction is None:
        instruction = "Given a web search query, retrieve relevant passages that answer the query"
    return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}"

def process_inputs(pairs, max_length=8192):
    prefix = '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
    suffix = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
    prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
    suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)
    inputs = tokenizer(
        pairs,
        padding=False,
        truncation="longest_first",
        return_attention_mask=False,
        max_length=max_length - len(prefix_tokens) - len(suffix_tokens),
    )
    for i, input_ids in enumerate(inputs["input_ids"]):
        inputs["input_ids"][i] = prefix_tokens + input_ids + suffix_tokens
    inputs = tokenizer.pad(inputs, padding=True, return_tensors="pt")
    return {key: value.to(model.device) for key, value in inputs.items()}

@torch.no_grad()
def compute_scores(inputs):
    token_false_id = tokenizer.convert_tokens_to_ids("no")
    token_true_id = tokenizer.convert_tokens_to_ids("yes")
    batch_scores = model(**inputs).logits[:, -1, :]
    true_vector = batch_scores[:, token_true_id]
    false_vector = batch_scores[:, token_false_id]
    batch_scores = torch.stack([false_vector, true_vector], dim=1)
    batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
    return batch_scores[:, 1].exp().tolist()

task = "Given a web search query, retrieve relevant passages that answer the query"
queries = ["What is the capital of China?", "What is the capital of China?"]
documents = ["The capital of China is Beijing.", "Bananas are yellow."]
pairs = [format_instruction(task, query, doc) for query, doc in zip(queries, documents)]
scores = compute_scores(process_inputs(pairs))
print("documents =", documents)
print("scores =", scores)
